# Assignment: 4  Clustering and PageRank 

**Part 1** k-center (Farthest-First Traversal) and k-means++ clustering on the UCI Spam dataset  
**Part 2:** Inverted-index-based web search engine with TF-IDF scoring  
**Part 3:** PageRank on PySpark RDDs with β = 0.8, 40 iterations

---

In [1]:
# Imports & Pyspark Init
import os, sys, time, random, math, re
from collections import defaultdict
import numpy as np


try:
    import findspark
    findspark.init()
except ImportError:
    pass

from pyspark.sql import SparkSession
from pyspark.mllib.linalg import Vectors, DenseVector

spark = (
    SparkSession.builder
    .appName("Clustering_and_PageRank")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")           
    .config("spark.sql.shuffle.partitions", "8")    
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

print(f"Spark version  : {spark.version}")
print(f"App name       : {sc.appName}")
print(f"Master         : {sc.master}")
print(f"Default parallelism: {sc.defaultParallelism}")

26/04/15 17:59:20 WARN Utils: Your hostname, zarvis resolves to a loopback address: 127.0.1.1; using 192.168.1.47 instead (on interface wlo1)
26/04/15 17:59:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/15 17:59:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version  : 3.5.8
App name       : Clustering_and_PageRank
Master         : local[*]
Default parallelism: 8


---
## Part 1: Clustering

**Algorithms implemented:**
- **readVectorsSeq(filename)** — load comma-separated feature vectors into `DenseVector` objects  
- **kcenter(P, k)** — Farthest-First Traversal (k-center), O(|P| × k)  
- **kmeansPP(P, k)** — k-means++ seeding, O(|P| × k)  
- **kmeansObj(P, C)** — average squared distance to nearest centre

**Dataset:** datasets/Q1-UCI Spam clustering/spambase.data — 4 601 points × 58 dimensions

In [20]:

# Part 1 – Helper: squared Euclidean distance between two Vectors
def sq_dist(a, b):
    """Squared Euclidean distance between two DenseVectors"""
    diff = np.array(a) - np.array(b)
    return float(np.dot(diff, diff))


def readVectorsSeq(filename):

    points = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                vals = [float(x) for x in line.split(',')]
                points.append(Vectors.dense(vals))
    return points

In [21]:

# kcenter  –  Farthest-First Traversal

def kcenter(P, k):

    n = len(P)
    # Deterministic start: pick the first point as the initial centre
    centers = [P[0]]

    # min_dist[i] = min squared distance from P[i] to any centre chosen so far
    min_dist = [sq_dist(P[i], centers[0]) for i in range(n)]

    for _ in range(k - 1):
        # O(|P|): find point farthest from all current centres
        max_idx = 0
        for i in range(1, n):
            if min_dist[i] > min_dist[max_idx]:
                max_idx = i

        new_center = P[max_idx]
        centers.append(new_center)

        # O(|P|): update min_dist with the newly added centre
        for i in range(n):
            d = sq_dist(P[i], new_center)
            if d < min_dist[i]:
                min_dist[i] = d

    return centers

In [4]:

# kmeansPP  –  k-means++ seeding


def kmeansPP(P, k):

    n = len(P)

    # Step 1: pick first centre uniformly at random
    first_idx = random.randint(0, n - 1)
    centers = [P[first_idx]]

    # D[i] = min squared distance from P[i] to any centre chosen so far
    D = [sq_dist(P[i], centers[0]) for i in range(n)]

    for _ in range(k - 1):
        # O(|P|): weighted random sample (D^2 sampling)
        total = sum(D)
        threshold = random.random() * total
        cumsum = 0.0
        next_idx = n - 1          # fallback: last point
        for i in range(n):
            cumsum += D[i]
            if cumsum >= threshold:
                next_idx = i
                break

        new_center = P[next_idx]
        centers.append(new_center)

        # O(|P|): update D with newly added centre
        for i in range(n):
            d = sq_dist(P[i], new_center)
            if d < D[i]:
                D[i] = d

    return centers

In [5]:

# kmeansObj  –  k-means objective (average squared distance)


def kmeansObj(P, C):

    total = 0.0
    for p in P:
        min_d = min(sq_dist(p, c) for c in C)
        total += min_d
    return total / len(P)

In [6]:

# Part 1 – Main Experiments  (k = 5, k1 = 20)


DATA_PATH = "datasets/Q1-UCI Spam clustering/spambase.data"
k  = 5
k1 = 20

print("Loading spambase dataset ...")
P = readVectorsSeq(DATA_PATH)
print(f"Loaded {len(P)} points × {len(P[0])} dimensions\n")

# Experiment 1: kcenter(P, k)
print(f"Experiment 1: kcenter(P, k={k})")
t0 = time.time()
C_kc = kcenter(P, k)
elapsed = time.time() - t0
print(f"Running time for kcenter(P, k={k}): {elapsed:.4f} seconds")

#Experiment 2: kmeansPP(P, k)
print(f"\nExperiment 2: kmeansPP(P, k={k})")
C_pp = kmeansPP(P, k)
obj2 = kmeansObj(P, C_pp)
print(f"kmeansObj for kmeansPP(P, k={k}): {obj2:.6f}")

#Experiment 3: coreset (kcenter → kmeansPP)
print(f"\nExperiment 3: Coreset Experiment")
print(f"kcenter(P, k1={k1})  →  kmeansPP(X, k={k})  →  kmeansObj(P, C)")
X        = kcenter(P, k1)          # k1-centre coreset
C_core   = kmeansPP(X, k)          # k centres chosen from coreset
obj3     = kmeansObj(P, C_core)    # evaluate on full dataset
print(f"kmeansObj for coreset experiment (kcenter k1={k1} → kmeansPP k={k}): {obj3:.6f}")

print("\n--- Interpretation ---")
print(f"Direct kmeansPP obj : {obj2:.6f}")
print(f"Coreset approach obj: {obj3:.6f}")

Loading spambase dataset ...
Loaded 4601 points × 58 dimensions

Experiment 1: kcenter(P, k=5)
Running time for kcenter(P, k=5): 0.2903 seconds

Experiment 2: kmeansPP(P, k=5)
kmeansObj for kmeansPP(P, k=5): 129874.719849

Experiment 3: Coreset Experiment
kcenter(P, k1=20)  →  kmeansPP(X, k=5)  →  kmeansObj(P, C)
kmeansObj for coreset experiment (kcenter k1=20 → kmeansPP k=5): 9537361.723089

--- Interpretation ---
Direct kmeansPP obj : 129874.719849
Coreset approach obj: 9537361.723089


## Part 2: Part 2: Inverted-index-based web search engine with TF-IDF scoring



In [7]:
# Part 2 – Constants and tokenisation helpers


WEBPAGES_FOLDER = "datasets/Q2-webSearch/webpages"

# Connector / stop words: count for positions but do NOT store
STOP_WORDS = frozenset({
    'a', 'an', 'the', 'they', 'these', 'this', 'for',
    'is', 'are', 'was', 'of', 'or', 'and', 'does', 'will', 'whose'
})

# Punctuation characters to replace with a space (exhaustive per spec)
PUNCT_CHARS = set('{}[]<>=(). ,;\'"?#!-:')

# Plural → singular normalisation map (exhaustive per spec)
PLURAL_MAP = {
    'stacks'       : 'stack',
    'structures'   : 'structure',
    'applications' : 'application'
}


def preprocess_text(text):

    # 1. Replace each punctuation char with a space
    buf = []
    for ch in text:
        buf.append(' ' if ch in PUNCT_CHARS else ch)
    text = ''.join(buf)

    # 2. Split on whitespace and lowercase each token
    raw_tokens = text.lower().split()

    # 3. Apply plural normalisation; check stop-word status
    result = []
    for tok in raw_tokens:
        normalised = PLURAL_MAP.get(tok, tok)
        result.append((normalised, normalised in STOP_WORDS))
    return result


def normalise_query_word(word):
 
    w = word.lower()
    return PLURAL_MAP.get(w, w)

In [8]:

# MySet


class MySet:

    def __init__(self):
        self._elements = []

    def addElement(self, element):
  
        if element not in self._elements:
            self._elements.append(element)

    def union(self, otherSet):    
        result = MySet()
        for e in self._elements:
            result.addElement(e)
        for e in otherSet._elements:
            result.addElement(e)
        return result

    def intersection(self, otherSet):
        result = MySet()
        for e in self._elements:
            if e in otherSet._elements:
                result.addElement(e)
        return result

    def __iter__(self):
        return iter(self._elements)

    def __len__(self):
        return len(self._elements)

    def __bool__(self):
        return bool(self._elements)

    def __contains__(self, item):
        return item in self._elements

In [9]:

# Position


class Position:

    def __init__(self, p, wordIndex):
 
        self._page  = p
        self._index = wordIndex

    def getPageEntry(self):
        return self._page

    def getWordIndex(self):
        return self._index

In [10]:

# WordEntry


class WordEntry:


    def __init__(self, word):
  
        self._word      = word
        self._positions = []   # list of Position objects

    @property
    def word(self):
        return self._word

    def addPosition(self, position):
  
        self._positions.append(position)

    def addPositions(self, positions):

        self._positions.extend(positions)

    def getAllPositionsForThisWord(self):
 
        return self._positions

    def getTermFrequency(self, page_name):
    
        count = 0
        target_page = None
        for pos in self._positions:
            if pos.getPageEntry().pageName == page_name:
                count += 1
                if target_page is None:
                    target_page = pos.getPageEntry()
        if count == 0 or target_page is None:
            return 0.0
        total = target_page._total_token_count
        return count / total if total > 0 else 0.0

In [11]:

# PageIndex


class PageIndex:
    

    def __init__(self):
        self._entries = {}   # normalised_word (str) → WordEntry

    def addPositionForWord(self, word, position):

        if word not in self._entries:
            self._entries[word] = WordEntry(word)
        self._entries[word].addPosition(position)

    def getWordEntries(self):
     
        return list(self._entries.values())

    def getWordEntry(self, word):
        return self._entries.get(word)

    def containsWord(self, word):
        return word in self._entries

    def getPositionsOfWord(self, word):
        entry = self._entries.get(word)
        if entry is None:
            return None
        return sorted(pos.getWordIndex() for pos in entry.getAllPositionsForThisWord())

In [12]:

# PageEntry


class PageEntry:

    def __init__(self, pageName):
  
        self.pageName         = pageName
        self._page_index      = PageIndex()
        self._total_token_count = 0   # ALL tokens (inc. stop words) for TF denominator
        self._readAndIndex()

    def _readAndIndex(self):
        filepath = os.path.join(WEBPAGES_FOLDER, self.pageName)
        try:
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as fh:
                content = fh.read()
        except FileNotFoundError:
            print(f"[WARNING] File not found: {filepath}")
            return

        tokens = preprocess_text(content)   # list of (normalised_word, is_stop)

        for pos_idx, (word, is_stop) in enumerate(tokens):
            position = pos_idx + 1          # 1-indexed, counts ALL tokens
            if word and not is_stop:        # skip stop words and empty strings
                self._page_index.addPositionForWord(word, Position(self, position))

        self._total_token_count = len(tokens)

    def getPageIndex(self):
        return self._page_index

In [13]:

# MyHashTable


class MyHashTable:


    TABLE_SIZE = 1031   # prime for good distribution

    def __init__(self):
        # Each bucket holds a list of WordEntry objects
        self._table = [[] for _ in range(self.TABLE_SIZE)]

    #hash function 
    def getHashIndex(self, word):

        h = 0
        for ch in word:
            h = (h * 31 + ord(ch)) % self.TABLE_SIZE
        return h

    # insert / merge 
    def addPositionsForWord(self, word_entry):
    
        word   = word_entry.word
        idx    = self.getHashIndex(word)
        bucket = self._table[idx]

        for existing in bucket:
            if existing.word == word:
                # Merge: append all positions from the incoming entry
                existing.addPositions(word_entry.getAllPositionsForThisWord())
                return

        # No existing entry found – create a new one and add to bucket
        new_entry = WordEntry(word)
        new_entry.addPositions(word_entry.getAllPositionsForThisWord())
        bucket.append(new_entry)

    #lookup
    def getWordEntry(self, word):
        """
        Retrieve the WordEntry for `word`, or None if not present.
        """
        idx    = self.getHashIndex(word)
        bucket = self._table[idx]
        for entry in bucket:
            if entry.word == word:
                return entry
        return None

In [14]:

# InvertedPageIndex

class InvertedPageIndex:


    def __init__(self):
        self._hash_table = MyHashTable()
        self._pages      = {}   # pageName (str) → PageEntry

    def addPage(self, page_entry):

        self._pages[page_entry.pageName] = page_entry
        for word_entry in page_entry.getPageIndex().getWordEntries():
            self._hash_table.addPositionsForWord(word_entry)

    def getPagesWhichContainWord(self, word):
        result     = MySet()
        word_entry = self._hash_table.getWordEntry(word)
        if word_entry is None:
            return result

        seen = set()
        for pos in word_entry.getAllPositionsForThisWord():
            page = pos.getPageEntry()
            if page.pageName not in seen:
                result.addElement(page)
                seen.add(page.pageName)
        return result

    def hasPage(self, pageName):
        """True iff the page has been added to the index."""
        return pageName in self._pages

    def getPage(self, pageName):
        """Return the PageEntry for `pageName`, or None."""
        return self._pages.get(pageName)



# SearchEngine


class SearchEngine:
   

    def __init__(self):    
        self._index = InvertedPageIndex()

    def performAction(self, actionMessage):

        parts  = actionMessage.strip().split()
        if not parts:
            return None
        action = parts[0]


        if action == "addPage":
            page_name = parts[1]
            self._index.addPage(PageEntry(page_name))
            return None           # no output per spec

     
        elif action == "queryFindPagesWhichContainWord":
            raw_word = parts[1]
            word     = normalise_query_word(raw_word)

            pages = self._index.getPagesWhichContainWord(word)
            if not pages:
                return f"No webpage contains word {raw_word}"
            # Return page names sorted alphabetically, comma+space separated
            return ", ".join(sorted(p.pageName for p in pages))

     
        elif action == "queryFindPositionsOfWordInAPage":
            raw_word  = parts[1]
            page_name = parts[2]

            # Is the page in the database?
            if not self._index.hasPage(page_name):
                return f"No webpage {page_name} found"

            word      = normalise_query_word(raw_word)
            positions = self._index.getPage(page_name).getPageIndex().getPositionsOfWord(word)

            if positions is None:
                return f"Webpage {page_name} does not contain word {raw_word}"
            return ", ".join(str(p) for p in positions)

        else:
            return f"Unknown action: {action}"

In [15]:
# Part 2 – Run all actions compare output with answers.txt


ACTIONS_FILE = "datasets/Q2-webSearch/actions.txt"
ANSWERS_FILE = "datasets/Q2-webSearch/answers.txt"

def run_and_compare():
    # Load actions and expected answers
    with open(ACTIONS_FILE, 'r') as fh:
        actions = [ln.strip() for ln in fh if ln.strip()]
    with open(ANSWERS_FILE, 'r') as fh:
        expected = [ln.strip() for ln in fh if ln.strip()]

    engine    = SearchEngine()
    ans_idx   = 0
    test_num  = 0
    all_pass  = True

 
    print(" Part 2: Action runner and correctness check")


    for action in actions:
        result = engine.performAction(action)

        if result is None:        
            continue

        # Query action: compare with expected answer
        test_num += 1
        exp = expected[ans_idx] if ans_idx < len(expected) else ""
        ans_idx += 1

        status = "PASS" if result == exp else "FAIL"
        if status == "FAIL":
            all_pass = False

        print(f"\nTest {test_num:2d}: {status}")
        print(f"  Action  : {action}")
        if status == "PASS":
            print(f"  Output  : {result}")
        else:
            print(f"  Expected: {exp}")
            print(f"  Got     : {result}")


    res = f"Congratulation!!! All {test_num} tests are Passed!" if all_pass else "Some tests FAILED – see details above."
    print("\n", res)



run_and_compare()

 Part 2: Action runner and correctness check

Test  1: PASS
  Action  : queryFindPagesWhichContainWord delhi
  Output  : No webpage contains word delhi

Test  2: PASS
  Action  : queryFindPagesWhichContainWord stack
  Output  : stack_datastructure_wiki

Test  3: PASS
  Action  : queryFindPagesWhichContainWord wikipedia
  Output  : stack_datastructure_wiki

Test  4: PASS
  Action  : queryFindPositionsOfWordInAPage magazines stack_datastructure_wiki
  Output  : Webpage stack_datastructure_wiki does not contain word magazines

Test  5: PASS
  Action  : queryFindPagesWhichContainWord allain
  Output  : No webpage contains word allain

Test  6: PASS
  Action  : queryFindPagesWhichContainWord allain
  Output  : stack_cprogramming

Test  7: PASS
  Action  : queryFindPagesWhichContainWord C
  Output  : stack_cprogramming

Test  8: PASS
  Action  : queryFindPagesWhichContainWord C++
  Output  : stack_cprogramming

Test  9: PASS
  Action  : queryFindPagesWhichContainWord jdk
  Output  : stack_or

## Part 3: PageRank on PySpark RDDs with β = 0.8, 40 iterations



In [16]:
# Part 3 – PageRank implementation on PySpark RDDs


def run_pagerank(filepath, beta=0.8, n_iter=40, verbose=True):


    # 1. Load and deduplicate edges
    raw_edges = sc.textFile(filepath)
    # Each line: "src dst"  (two integers)
    edges = raw_edges \
        .map(lambda line: tuple(map(int, line.strip().split()))) \
        .distinct()                     # remove duplicate (src, dst) pairs
    edges.cache()

    # 2. Discover all unique nodes
    all_nodes = edges.map(lambda e: e[0]).union(edges.map(lambda e: e[1])) \
                     .distinct()
    all_nodes.cache()
    n = all_nodes.count()

    if verbose:
        e_count = edges.count()
        print(f"Graph loaded: {n} nodes, {e_count} edges (after deduplication)")

    # 3. Build adjacency list with edge weights 

    def make_weighted_adj(dsts):
        dst_list = list(dsts)
        w = 1.0 / len(dst_list)
        return [(d, w) for d in dst_list]

    adj = edges.groupByKey().mapValues(make_weighted_adj)
    adj.cache()

    # 4. Initialise ranks
    init_rank = 1.0 / n
    ranks     = all_nodes.map(lambda v: (v, init_rank))
    teleport  = (1.0 - beta) / n

    #5. Iterative update
    for iteration in range(n_iter):
        # Each source node contributes rank * weight to each neighbour
        # adj.join(ranks): (src, ([(dst,w),...], rank))
        contribs = adj.join(ranks).flatMap(
            lambda x: [(dst, x[1][1] * w) for dst, w in x[1][0]]
        )

        # Sum incoming contributions per destination
        sum_contribs = contribs.reduceByKey(lambda a, b: a + b)

        # New rank = teleport_constant + beta * sum_contributions
        # leftOuterJoin guarantees every node
        # receives at least the teleport contribution.
        ranks = (
            all_nodes.map(lambda v: (v, teleport))
            .leftOuterJoin(sum_contribs)
            .map(lambda x: (
                x[0],
                x[1][0] + beta * (x[1][1] if x[1][1] is not None else 0.0)
            ))
        )

        # Materialise every 10 iterations to prevent DAG lineage explosion
        if (iteration + 1) % 10 == 0:
            ranks = sc.parallelize(ranks.collect())
            if verbose:
                print(f"  ... iteration {iteration + 1} / {n_iter} done")

    #6. Collect and sort
    results = ranks.collect()
    results.sort(key=lambda x: x[1], reverse=True)
    return results

In [17]:

# Part 3 – Validation on small.txt  (53 nodes)

SMALL_PATH = "datasets/Q3-PageRank/small.txt"


print(" PageRank validation on small.txt (53 nodes)")


small_results = run_pagerank(SMALL_PATH, beta=0.8, n_iter=40, verbose=True)

top_score = small_results[0][1]
status    = "PASS" if abs(top_score - 0.036) < 0.002 else "CHECK"
print(f"\nTop score: {top_score:.4f}  (expected ~0.036)  [{status}]")

print("\nTop 5 nodes:")
for node, score in small_results[:5]:
    print(f"  Node {node:4d}  score = {score:.6f}")

print("\nBottom 5 nodes:")
for node, score in small_results[-5:]:
    print(f"  Node {node:4d}  score = {score:.6f}")

 PageRank validation on small.txt (53 nodes)
Graph loaded: 100 nodes, 950 edges (after deduplication)
  ... iteration 10 / 40 done
  ... iteration 20 / 40 done
  ... iteration 30 / 40 done
  ... iteration 40 / 40 done

Top score: 0.0357  (expected ~0.036)  [PASS]

Top 5 nodes:
  Node   53  score = 0.035731
  Node   14  score = 0.034171
  Node   40  score = 0.033630
  Node    1  score = 0.030006
  Node   27  score = 0.029720

Bottom 5 nodes:
  Node   89  score = 0.003922
  Node   37  score = 0.003808
  Node   81  score = 0.003695
  Node   59  score = 0.003670
  Node   85  score = 0.003410


In [18]:
# Part 3 – Full run on whole.txt  (1 000 nodes, 8 192 edges)


WHOLE_PATH = "datasets/Q3-PageRank/whole.txt"


print(" \n PageRank on whole.txt (1 000 nodes)")


whole_results = run_pagerank(WHOLE_PATH, beta=0.8, n_iter=40, verbose=True)

print("\nTop 5 nodes with highest PageRank:")
print(f"  {'Node':>6}   {'Score':>12}")
print(f"  {'------':>6}   {'------------':>12}")
for node, score in whole_results[:5]:
    print(f"  {node:6d}   {score:.8f}")

print("\nBottom 5 nodes with lowest PageRank:")
print(f"  {'Node':>6}   {'Score':>12}")
print(f"  {'------':>6}   {'------------':>12}")
for node, score in whole_results[-5:]:
    print(f"  {node:6d}   {score:.8f}")

print("\n--- Summary ---")
scores = [s for _, s in whole_results]
print(f"Max rank  : {max(scores):.8f}")
print(f"Min rank  : {min(scores):.8f}")
print(f"Mean rank : {sum(scores)/len(scores):.8f}  (should be ≈ 1/1000 = 0.001)")

 
 PageRank on whole.txt (1 000 nodes)
Graph loaded: 1000 nodes, 8161 edges (after deduplication)
  ... iteration 10 / 40 done
  ... iteration 20 / 40 done
  ... iteration 30 / 40 done
  ... iteration 40 / 40 done

Top 5 nodes with highest PageRank:
    Node          Score
  ------   ------------
     263   0.00202029
     537   0.00194334
     965   0.00192545
     243   0.00185263
     285   0.00182737

Bottom 5 nodes with lowest PageRank:
    Node          Score
  ------   ------------
     408   0.00038780
     424   0.00035482
      62   0.00035315
      93   0.00035136
     558   0.00032860

--- Summary ---
Max rank  : 0.00202029
Min rank  : 0.00032860
Mean rank : 0.00100000  (should be ≈ 1/1000 = 0.001)


In [19]:
# ── Clean up Spark session ────────────────────────────────────
spark.stop()
print("SparkSession stopped.")

SparkSession stopped.
